# Explore simulation run

Load tick log and final world snapshot; plot population, organic heatmap, and concept counts.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "sim-core").exists():
    ROOT = ROOT.parent

EXPORTS = ROOT / "exports"
TICK_LOG = EXPORTS / "logs" / "tick_log.jsonl"
SNAPSHOT = EXPORTS / "snapshots" / "world_final.json"

print("Tick log:", TICK_LOG)
print("Snapshot:", SNAPSHOT)

In [ ]:
def load_tick_log(path: Path) -> list[dict]:
    entries = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

def load_snapshot(path: Path) -> dict:
    with path.open() as f:
        return json.load(f)

entries = load_tick_log(TICK_LOG) if TICK_LOG.exists() else []
snapshot = load_snapshot(SNAPSHOT) if SNAPSHOT.exists() else {}
print(f"Loaded {len(entries)} tick log entries, snapshot time={snapshot.get('time')}")

In [ ]:
if entries:
    pop = [len(e.get("creatures", [])) for e in entries]
    ticks = [e.get("tick", i) for i, e in enumerate(entries)]
    births = [len(e.get("births", [])) for e in entries]
    deaths = [len(e.get("deaths", [])) for e in entries]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ticks, pop, label="population")
    ax.plot(ticks, births, alpha=0.5, label="births/tick")
    ax.plot(ticks, deaths, alpha=0.5, label="deaths/tick")
    ax.set_xlabel("tick")
    ax.set_ylabel("count")
    ax.set_title("Population over time")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No tick log — run: cargo run -- --ticks 1000 --seed 42 --world-size 4 --creatures 12 --output exports")

In [ ]:
chunks = snapshot.get("chunks", [])
if chunks:
    c0 = chunks[0]
    organic = c0.get("organic", [])
    if organic:
        import numpy as np
        grid = np.array(organic)
        fig, ax = plt.subplots(figsize=(5, 5))
        im = ax.imshow(grid.T, origin="lower", cmap="YlGn")
        ax.set_title(f"Organic slice z={c0.get('slice_z')} chunk {c0.get('coord')}")
        plt.colorbar(im, ax=ax, label="organic")
        plt.tight_layout()
        plt.show()
else:
    print("No chunk data in snapshot")

In [ ]:
creatures = snapshot.get("creatures", [])
if creatures:
    df = pd.DataFrame([
        {
            "id": c["id"],
            "concept_count": c.get("concept_count", 0),
            "trusted_signatures": c.get("trusted_signature_count", 0),
            "memory_nodes": c.get("memory_node_count", 0),
        }
        for c in creatures
    ])
    display(df.describe())

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(df["id"].astype(str), df["concept_count"], label="concepts")
    if df["trusted_signatures"].sum() > 0:
        ax.bar(df["id"].astype(str), df["trusted_signatures"], alpha=0.6, label="trusted signatures")
    ax.set_xlabel("creature id")
    ax.set_ylabel("count")
    ax.set_title("Concept and trusted-signature counts (final snapshot)")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No creatures in snapshot")

In [ ]:
NARRATIVE = EXPORTS / "logs" / "narrative_summary.json"

def load_narrative(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open() as f:
        return json.load(f)

narrative = load_narrative(NARRATIVE)
print("Narrative keys:", list(narrative.keys()) if narrative else "(missing)")

In [ ]:
if entries:
    concept_pop = [
        sum(c.get("concept_count", 0) for c in e.get("creatures", []))
        for e in entries
    ]
    displacement = [e.get("mean_displacement", 0.0) for e in entries]
    novel_frac = [e.get("novel_sensor_fraction", 0.0) for e in entries]
    imagination = [e.get("imagination_events", 0) for e in entries]

    fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
    axes[0, 0].plot(ticks, concept_pop, color="C5")
    axes[0, 0].set_ylabel("live concepts")
    axes[0, 0].set_title("Population concept count over time")

    axes[0, 1].plot(ticks, displacement, color="C0")
    axes[0, 1].set_ylabel("mean displacement")
    axes[0, 1].set_title("Exploration displacement per tick")

    axes[1, 0].plot(ticks, novel_frac, color="C3")
    axes[1, 0].set_ylabel("novel sensor fraction")
    axes[1, 0].set_xlabel("tick")
    axes[1, 0].set_title("Novel sensory ticks")

    axes[1, 1].plot(ticks, imagination, color="C4")
    axes[1, 1].set_ylabel("imagination events")
    axes[1, 1].set_xlabel("tick")
    axes[1, 1].set_title("Sleep imagination replay")
    plt.tight_layout()
    plt.show()
else:
    print("No tick log for exploration/concept-over-time plots")

In [ ]:
if entries:
    sounds = [e.get("sound_event_count", 0) for e in entries]
    transfers = [e.get("transfer_count", e.get("action_counts", {}).get("transfer_organic_count", 0)) for e in entries]
    concepts_tick = [e.get("concepts_formed", 0) for e in entries]

    fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    axes[0].plot(ticks, sounds, color="C2")
    axes[0].set_ylabel("sound events")
    axes[0].set_title("Sound events per tick")

    axes[1].plot(ticks, transfers, color="C4")
    axes[1].set_ylabel("transfers")
    axes[1].set_title("TransferOrganic count per tick")

    axes[2].plot(ticks, concepts_tick, color="C5")
    axes[2].set_xlabel("tick")
    axes[2].set_ylabel("concepts formed")
    axes[2].set_title("Concept formation per tick")
    plt.tight_layout()
    plt.show()

    print(f"Total sounds: {sum(sounds)}, transfers: {sum(transfers)}, concepts formed: {sum(concepts_tick)}")
else:
    print("No tick log for sound/transfer/concept plots")

In [ ]:
if narrative:
    events = narrative.get("events", [])
    concept_events = [e for e in events if e.get("kind") == "concept_spike"]
    print(
        f"Narrative: births={narrative.get('total_births')}, deaths={narrative.get('total_deaths')}, "
        f"concepts_formed={narrative.get('total_concepts_formed')}, "
        f"population_concepts={narrative.get('population_concept_count')}, "
        f"concept_spikes={narrative.get('concept_spike_count')}"
    )
    if concept_events:
        import pandas as pd
        display(pd.DataFrame(concept_events).head(10))
    else:
        print("No concept_spike events in narrative")
else:
    print("No narrative_summary.json — run simulation with export")